In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade peft trl datasets accelerate bitsandbytes

In [ ]:
# ============================================================
# Cell 1: Install (run once after fresh runtime)
# ============================================================
# !pip install -q git+https://github.com/huggingface/transformers.git
# !pip install -q --upgrade peft trl datasets accelerate bitsandbytes

# ============================================================
# Cell 2: DPO Training (fixed — pre-computed reference)
# ============================================================
import os
import json
import random
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- Config ---
MODEL_ID = "Qwen/Qwen3.5-4B"
SFT_ADAPTER_PATH = "/kaggle/input/datasets/yuanmazax/delta-filing-sft-result"
DPO_DATA_PATH = "/kaggle/input/datasets/yuanmazax/delta-filing-dpo/dpo_pytorch.jsonl"
OUTPUT_DIR = "/kaggle/working/adapters/dpo_pytorch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BETA = 0.1
LR = 5e-5
EPOCHS = 2
LOG_EVERY = 10
EVAL_EVERY = 50
SAVE_EVERY = 100
MAX_SEQ_LEN = 1024

SIMPLE_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{{ message['content'] | trim }}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

SYSTEM_PROMPT = (
    "You are Delta Filing, a financial analyst AI specializing in SEC filing analysis. "
    "You analyze 10-K and 10-Q filings, detect year-over-year changes in risk disclosures, "
    "track management guidance accuracy, and flag potential red flags. "
    "Always reference specific filing sections, cite specific numbers and dates, "
    "provide analytical judgment, and note caveats. "
    "Do not give investment advice or predict stock prices."
)

TOOL_SYSTEM_PROMPT = (
    "You are Delta Filing, a financial analyst AI specializing in SEC filing analysis. "
    "You have access to tools for retrieving SEC filings, financial data, news, and insider trades. "
    "When you need data, call the appropriate tool. When you have data, analyze it thoroughly.\n\n"
    "Available tools:\n\n"
    "1. search_filings(ticker, filing_type, count)\n"
    "2. get_filing_section(ticker, section_id, filing_type)\n"
    "3. diff_filing_sections(ticker, section_id, filing_type)\n"
    "4. stock_price(ticker, period)\n"
    "5. company_metrics(ticker)\n"
    "6. company_news(ticker, days)\n"
    "7. insider_trades(ticker)\n"
    "8. analyst_ratings(ticker)\n\n"
    'To use a tool, respond with JSON: {"tool": "name", "arguments": {...}}\n'
    "Respond with ONLY the JSON, nothing else."
)

print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

# --- Tokenizer ---
print("\n[1] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load DPO data ---
print("\n[2] Loading DPO data...")
with open(DPO_DATA_PATH) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

pairs = []
for ex in raw_data:
    category = ex.get("category", "analysis")
    sys_prompt = TOOL_SYSTEM_PROMPT if category == "tool_calling" else SYSTEM_PROMPT

    chosen_text = tokenizer.apply_chat_template([
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["chosen"]},
    ], tokenize=False)
    rejected_text = tokenizer.apply_chat_template([
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["rejected"]},
    ], tokenize=False)

    chosen_ids = tokenizer.encode(chosen_text, max_length=MAX_SEQ_LEN, truncation=True)
    rejected_ids = tokenizer.encode(rejected_text, max_length=MAX_SEQ_LEN, truncation=True)

    pairs.append({
        "chosen_ids": torch.tensor(chosen_ids, dtype=torch.long),
        "rejected_ids": torch.tensor(rejected_ids, dtype=torch.long),
    })

random.seed(42)
random.shuffle(pairs)
split = int(len(pairs) * 0.85)
train_pairs = pairs[:split]
valid_pairs = pairs[split:]
print(f"  Total: {len(pairs)}, Train: {len(train_pairs)}, Valid: {len(valid_pairs)}")

# --- Load model with SFT adapter ---
print("\n[3] Loading model with SFT adapter...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    trust_remote_code=True, device_map={"": 0},
)
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH, is_trainable=True)

memory_gb = base_model.get_memory_footprint() / 1e9
print(f"  Memory: {memory_gb:.1f} GB")
assert memory_gb < 5.0, f"Quantization not working: {memory_gb:.1f} GB"


# ============================================================
#  Log probability computation
# ============================================================

def compute_sequence_log_probs(model, input_ids):
    """Compute total log probability of a token sequence."""
    x = input_ids.unsqueeze(0).to("cuda:0")
    with torch.amp.autocast("cuda", dtype=torch.float16):
        outputs = model(x)
        logits = outputs.logits
    shift_logits = logits[:, :-1, :].float()
    shift_labels = x[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = torch.gather(
        log_probs, dim=-1,
        index=shift_labels.unsqueeze(-1),
    ).squeeze(-1)
    total = token_log_probs.sum()
    del logits, shift_logits, log_probs, token_log_probs, outputs
    return total


# ============================================================
#  Pre-compute ALL reference log probs (with SFT adapter)
# ============================================================

print("\n[4] Pre-computing reference log probs (this IS the SFT model)...")
model.eval()

ref_cache = []
with torch.no_grad():
    for i, pair in enumerate(pairs):
        ref_chosen = compute_sequence_log_probs(model, pair["chosen_ids"]).item()
        ref_rejected = compute_sequence_log_probs(model, pair["rejected_ids"]).item()
        ref_cache.append({"ref_chosen": ref_chosen, "ref_rejected": ref_rejected})

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(pairs)} done")
        if (i + 1) % 100 == 0:
            torch.cuda.empty_cache()

print(f"  Cached {len(ref_cache)} reference pairs")

# Sanity check: reference margin should be moderate
sample_margins = [c["ref_chosen"] - c["ref_rejected"] for c in ref_cache[:20]]
avg_margin = sum(sample_margins) / len(sample_margins)
print(f"  Avg ref margin (chosen - rejected): {avg_margin:.2f}")

# Split cache same way as pairs
ref_train = ref_cache[:split]
ref_valid = ref_cache[split:]


# ============================================================
#  Now enable training
# ============================================================

model.gradient_checkpointing_enable()
model.train()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Trainable params: {trainable:,}")
print(f"  GPU: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB allocated")


# ============================================================
#  DPO Loss (uses cached reference values)
# ============================================================

def dpo_loss(model, chosen_ids, rejected_ids, ref_chosen, ref_rejected, beta):
    """DPO loss with pre-computed reference log probs.

    Policy forward passes get gradients.
    Reference values are pre-computed floats — no second model needed.

    Loss = -log(sigmoid(β × ((π_chosen - ref_chosen) - (π_rejected - ref_rejected))))
    """
    # Policy log probs (gets gradients)
    pi_chosen = compute_sequence_log_probs(model, chosen_ids)
    pi_rejected = compute_sequence_log_probs(model, rejected_ids)

    # Rewards using cached reference values
    chosen_reward = pi_chosen - ref_chosen
    rejected_reward = pi_rejected - ref_rejected

    # DPO loss
    reward_margin = beta * (chosen_reward - rejected_reward)
    loss = -F.logsigmoid(reward_margin)

    return loss, {
        "loss": loss.item(),
        "reward_margin": reward_margin.item(),
        "chosen_reward": chosen_reward.item(),
        "rejected_reward": rejected_reward.item(),
    }


# ============================================================
#  Training loop
# ============================================================

print(f"\n[5] Starting DPO training...")
print(f"  β={BETA}, LR={LR}, Epochs={EPOCHS}")

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)

global_step = 0
best_val_loss = float("inf")
train_indices = list(range(len(train_pairs)))

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    random.shuffle(train_indices)
    losses, margins = [], []

    for i, idx in enumerate(train_indices):
        pair = train_pairs[idx]
        ref = ref_train[idx]

        optimizer.zero_grad()
        loss, metrics = dpo_loss(
            model,
            pair["chosen_ids"], pair["rejected_ids"],
            ref["ref_chosen"], ref["ref_rejected"],
            BETA,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        del loss
        torch.cuda.empty_cache()

        losses.append(metrics["loss"])
        margins.append(metrics["reward_margin"])
        global_step += 1

        if global_step % LOG_EVERY == 0:
            avg_l = sum(losses[-LOG_EVERY:]) / LOG_EVERY
            avg_m = sum(margins[-LOG_EVERY:]) / LOG_EVERY
            mem = torch.cuda.memory_allocated(0) / 1e9
            print(f"  Step {global_step:4d} | loss={avg_l:.4f} | "
                  f"margin={avg_m:.3f} | mem={mem:.1f}GB")

        if global_step % EVAL_EVERY == 0:
            model.eval()
            vl, vm = [], []
            with torch.no_grad():
                for vi in range(min(20, len(valid_pairs))):
                    vp = valid_pairs[vi]
                    rv = ref_valid[vi]
                    l, m = dpo_loss(
                        model,
                        vp["chosen_ids"], vp["rejected_ids"],
                        rv["ref_chosen"], rv["ref_rejected"],
                        BETA,
                    )
                    vl.append(m["loss"])
                    vm.append(m["reward_margin"])
            avg_vl = sum(vl) / len(vl)
            avg_vm = sum(vm) / len(vm)
            is_best = avg_vl < best_val_loss
            if is_best:
                best_val_loss = avg_vl
            print(f"  Step {global_step:4d} | VAL loss={avg_vl:.4f} | "
                  f"margin={avg_vm:.3f}{'  ★' if is_best else ''}")
            model.train()

        if global_step % SAVE_EVERY == 0:
            ckpt = os.path.join(OUTPUT_DIR, f"checkpoint-{global_step}")
            model.save_pretrained(ckpt)
            print(f"  Step {global_step:4d} | Saved {ckpt}")

# --- Save final ---
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n{'='*60}")
print(f"  DPO complete! Best val loss: {best_val_loss:.4f}")
print(f"  Saved to: {OUTPUT_DIR}")
print(f"{'='*60}")